In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict, train_test_split #See how the split is done
from sklearn.metrics import confusion_matrix

In [2]:
data = pd.read_csv('Titanic-Dataset.csv')

In [ ]:
# train, validate, test = \
#               np.split(data.sample(frac=1, random_state=42), 
#                        [int(.7*len(data)), int(.85*len(data))])

# y_train = train['Survived']
# x_train = train.drop(columns = ['Survived'])

# y_val = validate['Survived']
# x_val = validate.drop(columns = ['Survived'])

# y_test = test['Survived']
# x_test = test.drop(columns = ['Survived'])

In [3]:
X = data.drop(columns=['Survived'])
y = data['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

numeric_features = ['Age', 'Fare']
X_train[numeric_features] = X_train[numeric_features].fillna(X_train[numeric_features].median())
X_test[numeric_features] = X_test[numeric_features].fillna(X_test[numeric_features].median())

sc = StandardScaler()
X_train[numeric_features] = sc.fit_transform(X_train[numeric_features])
X_test[numeric_features] = sc.transform(X_test[numeric_features])

categorical_features = ['Sex', 'Embarked']
X_train[categorical_features] = X_train[categorical_features].fillna(X_train[categorical_features].mode().iloc[0])
X_test[categorical_features] = X_test[categorical_features].fillna(X_train[categorical_features].mode().iloc[0])

X_train = pd.get_dummies(X_train, columns = categorical_features, dtype=int)
X_test = pd.get_dummies(X_test, columns = categorical_features, dtype=int)

X_train.drop(columns = ['Name', 'PassengerId',  'Cabin','SibSp', 'Parch', 'Ticket'], inplace = True)
X_test.drop(columns = ['Name', 'PassengerId',  'Cabin','SibSp', 'Parch', 'Ticket'], inplace = True)

In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

model = RandomForestClassifier(n_estimators=10, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='accuracy')
print(f'Cross-validation accuracy scores: {scores}')
print(f'Average Accuracy: {np.mean(scores):.4f}')

Cross-validation accuracy scores: [0.78289474 0.74342105 0.78807947 0.76821192 0.78807947]
Average Accuracy: 0.7741


In [5]:
model.fit(X_train, y_train)
test_accuracy = model.score(X_test, y_test)
print(f"Test set accuracy: {test_accuracy:.4f}")


train_accuracy = model.score(X_train, y_train)
print(f"Train set accuracy: {train_accuracy:.4f}")

Test set accuracy: 0.8284
Train set accuracy: 0.9498


In [6]:
confusion_matrix(y_test, model.predict(X_test))

array([[66, 12],
       [11, 45]])